<a href="https://colab.research.google.com/github/CodeCrusaders26/privacy-dns-filter/blob/feature%2Fml-model/notebooks/preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files

uploaded = files.upload()

print("Uploaded:")
for f in uploaded:
    print(f)

KeyboardInterrupt: 

In [ ]:
import pandas as pd
from io import StringIO

# 1. Tranco
tranco = pd.read_csv(
    "/content/drive/MyDrive/Colab Notebooks/Dataset/top-1m.csv",
    header=None,
    names=["rank", "domain"]
)

# Take the top 10,000 candidates
tranco = tranco.head(10000)

# 2. URLhaus
with open("/content/drive/MyDrive/Colab Notebooks/Dataset/csv.txt", "r", encoding="utf-8-sig") as f:
    lines = f.readlines()

header_index = next(
    i for i, line in enumerate(lines)
    if line.lstrip("# ").startswith("id,dateadded,url,")
)

urlhaus_text = (
    lines[header_index].lstrip("# ")
    + "".join(lines[header_index + 1:])
)

urlhaus = pd.read_csv(StringIO(urlhaus_text))

# 3. PhishTank
phishtank = pd.read_csv(
    "/content/drive/MyDrive/Colab Notebooks/Dataset/verified_online.csv",
    encoding="utf-8-sig"
)

# Display columns
print("TRANCO:", tranco.columns.tolist())
print("URLHAUS:", urlhaus.columns.tolist())
print("PHISHTANK:", phishtank.columns.tolist())

print("\nDataset sizes:")
print("Tranco:", len(tranco))
print("URLhaus:", len(urlhaus))
print("PhishTank:", len(phishtank))

TRANCO: ['rank', 'domain']
URLHAUS: ['id', 'dateadded', 'url', 'url_status', 'last_online', 'threat', 'tags', 'urlhaus_link', 'reporter']
PHISHTANK: ['phish_id', 'url', 'phish_detail_url', 'submission_time', 'verified', 'verification_time', 'online', 'target']

Dataset sizes:
Tranco: 10000
URLhaus: 168557
PhishTank: 46808


In [ ]:
import pandas as pd
import re
import ipaddress
from urllib.parse import urlsplit
from datetime import date

def extract_domain(value):
    if pd.isna(value):
        return None

    value = str(value).strip()

    try:
        if "://" not in value:
            value = "//" + value

        host = urlsplit(value).hostname

        if not host:
            return None

        host = host.lower().rstrip(".")

        # Exclude IP addresses
        try:
            ipaddress.ip_address(host)
            return None
        except ValueError:
            pass

        # Validate hostname
        host = host.encode("idna").decode("ascii")

        if len(host) > 253:
            return None

        if not re.fullmatch(
            r"(?=.{1,253}$)(?:[a-z0-9](?:[a-z0-9-]{0,61}[a-z0-9])?\.)+[a-z]{2,63}",
            host
        ):
            return None

        return host

    except (ValueError, UnicodeError):
        return None

In [ ]:
safe = tranco[["domain"]].copy()

safe["domain"] = safe["domain"].apply(extract_domain)
safe = safe.dropna(subset=["domain"])

safe["label"] = 0
safe["source"] = "Tranco"

safe = safe.drop_duplicates(subset=["domain"])

print("Legitimate domains:", len(safe))

Legitimate domains: 9999


In [ ]:
# URLhaus
uh = urlhaus[["url"]].copy()

uh["domain"] = uh["url"].apply(extract_domain)
uh = uh.dropna(subset=["domain"])

uh["label"] = 1
uh["source"] = "URLhaus"

# PhishTank: retain verified phishing records
pt = phishtank[
    phishtank["verified"].astype(str).str.lower().eq("yes")
][["url"]].copy()

pt["domain"] = pt["url"].apply(extract_domain)
pt = pt.dropna(subset=["domain"])

pt["label"] = 1
pt["source"] = "PhishTank"

# Combine malicious sources
malicious = pd.concat([
    uh[["domain", "label", "source"]],
    pt[["domain", "label", "source"]]
], ignore_index=True)

# Combine sources for domains appearing in both databases
malicious = (
    malicious.groupby("domain")
    .agg(
        label=("label", "max"),
        source=("source", lambda x: "+".join(sorted(set(x))))
    )
    .reset_index()
)

print("Unique malicious domains:", len(malicious))

Unique malicious domains: 21644


In [ ]:
# Remove any malicious domains from legitimate candidates
safe = safe[~safe["domain"].isin(malicious["domain"])]

# Select up to 10,000 malicious domains
malicious_sample = malicious.sample(
    n=min(len(malicious), 10000),
    random_state=42
)

# Combine
combined = pd.concat(
    [safe, malicious_sample],
    ignore_index=True
)

combined["collection_date"] = date.today().isoformat()

# Shuffle records
combined = combined.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# Export
combined.to_csv("combined_domains.csv", index=False)

print("Final dataset:", len(combined))
print("\nClass distribution:")
print(combined["label"].value_counts())

print("\nSample:")
print(combined.head(10))

files.download("combined_domains.csv")

Final dataset: 19950

Class distribution:
label
1    10000
0     9950
Name: count, dtype: int64

Sample:
                           domain  label     source collection_date
0                  findmykids.org      0     Tranco      2026-11-11
1                     labs.google      0     Tranco      2026-11-11
2  integrating.smbc-flexibler.com      1  PhishTank      2026-11-11
3                         unl.edu      0     Tranco      2026-11-11
4            ggi.dietondemand.com      1    URLhaus      2026-11-11
5                          bnf.fr      0     Tranco      2026-11-11
6                      qualys.com      0     Tranco      2026-11-11
7                   t.yesware.com      1  PhishTank      2026-11-11
8                   tagesschau.de      0     Tranco      2026-11-11
9                   flashscore.ua      0     Tranco      2026-11-11


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Validating  combined_domains.csv

In [1]:
import pandas as pd

df = pd.read_csv("/content/combined_domains.csv")

print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nDataset information:")
df.info()

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

Dataset shape: (19950, 4)

Columns:
['domain', 'label', 'source', 'collection_date']

First 5 rows:


,domain,label,source,collection_date
0,findmykids.org,0,Tranco,2026-11-11
1,labs.google,0,Tranco,2026-11-11
2,integrating.smbc-flexibler.com,1,PhishTank,2026-11-11
3,unl.edu,0,Tranco,2026-11-11
4,ggi.dietondemand.com,1,URLhaus,2026-11-11



Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19950 entries, 0 to 19949
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   domain           19950 non-null  object
 1   label            19950 non-null  int64 
 2   source           19950 non-null  object
 3   collection_date  19950 non-null  object
dtypes: int64(1), object(3)
memory usage: 623.6+ KB

Missing values:
domain             0
label              0
source             0
collection_date    0
dtype: int64

Duplicate rows:
0


In [2]:
print(df["label"].value_counts())

label
1    10000
0     9950
Name: count, dtype: int64


In [4]:
import pandas as pd

def extract_features(domain):
    domain = str(domain).lower().strip()

    return {
        "length": len(domain),
        "dots": domain.count("."),
        "hyphens": domain.count("-"),
        "digits": sum(c.isdigit() for c in domain),
        "special_chars": sum(
            not c.isalnum() and c not in ".-"
            for c in domain
        ),
        "subdomains": max(0, domain.count(".") - 1)
    }

features = pd.DataFrame(
    df["domain"].fillna("").apply(extract_features).tolist()
)

print(features.head())

   length  dots  hyphens  digits  special_chars  subdomains
0      14     1        0       0              0           0
1      11     1        0       0              0           0
2      30     2        1       0              0           1
3       7     1        0       0              0           0
4      20     2        0       0              0           1


In [5]:
processed_df = pd.concat(
    [features, df[["label"]].reset_index(drop=True)],
    axis=1
)

print(processed_df.head())
print(processed_df.shape)

   length  dots  hyphens  digits  special_chars  subdomains  label
0      14     1        0       0              0           0      0
1      11     1        0       0              0           0      0
2      30     2        1       0              0           1      1
3       7     1        0       0              0           0      0
4      20     2        0       0              0           1      1
(19950, 7)


In [6]:
processed_df.to_csv(
    "processed_domains.csv",
    index=False
)

In [7]:
# 1. Check label distribution
print("Label distribution:")
print(processed_df["label"].value_counts())

print("\nLabel percentages:")
print(processed_df["label"].value_counts(normalize=True) * 100)

# 2. Check missing values
print("\nMissing values:")
print(processed_df.isnull().sum())

# 3. Check duplicate rows
print("\nDuplicate rows:")
print(processed_df.duplicated().sum())

# 4. Check duplicate domains
print("\nDuplicate domains:")
print(df["domain"].duplicated().sum())

# 5. Check feature statistics
print("\nFeature statistics:")
display(processed_df.describe())

Label distribution:
label
1    10000
0     9950
Name: count, dtype: int64

Label percentages:
label
1    50.125313
0    49.874687
Name: proportion, dtype: float64

Missing values:
length           0
dots             0
hyphens          0
digits           0
special_chars    0
subdomains       0
label            0
dtype: int64

Duplicate rows:
18058

Duplicate domains:
0

Feature statistics:


,length,dots,hyphens,digits,special_chars,subdomains,label
count,19950.000000,19950.000000,19950.000000,19950.000000,19950.0,19950.000000,19950.000000
mean,19.139549,1.484160,0.374035,1.635238,0.0,0.484160,0.501253
std,11.083407,0.588479,0.797979,4.402722,0.0,0.588479,0.500011
min,4.000000,1.000000,0.000000,0.000000,0.0,0.000000,0.000000
25%,11.000000,1.000000,0.000000,0.000000,0.0,0.000000,0.000000
50%,15.000000,1.000000,0.000000,0.000000,0.0,0.000000,1.000000
75%,25.000000,2.000000,1.000000,1.000000,0.0,1.000000,1.000000
max,104.000000,8.000000,11.000000,50.000000,0.0,7.000000,1.000000


In [8]:
import math
from collections import Counter

def advanced_features(domain):
    domain = str(domain).lower().strip()

    counts = Counter(domain)

    entropy = -sum(
        (count / len(domain)) * math.log2(count / len(domain))
        for count in counts.values()
    ) if domain else 0

    return {
        "vowel_count": sum(c in "aeiou" for c in domain),
        "consonant_count": sum(c.isalpha() and c not in "aeiou" for c in domain),
        "unique_chars": len(set(domain)),
        "entropy": entropy,
        "longest_label": max(
            (len(part) for part in domain.split(".")),
            default=0
        ),
        "has_suspicious_words": int(
            any(word in domain for word in
                ["login", "verify", "secure", "update", "account", "bank"])
        )
    }

advanced_df = pd.DataFrame(
    df["domain"].apply(advanced_features).tolist()
)

processed_df = pd.concat(
    [features.reset_index(drop=True),
     advanced_df.reset_index(drop=True),
     df[["label"]].reset_index(drop=True)],
    axis=1
)

display(processed_df.head())
print("New shape:", processed_df.shape)

,length,dots,hyphens,digits,special_chars,subdomains,vowel_count,consonant_count,unique_chars,entropy,longest_label,has_suspicious_words,label
0,14,1,0,0,0,0,3,10,12,3.521641,10,0,0
1,11,1,0,0,0,0,4,6,8,2.913977,6,0,0
2,30,2,1,0,0,1,8,19,17,3.989898,14,0,1
3,7,1,0,0,0,0,3,3,6,2.521641,3,0,0
4,20,2,0,0,0,1,7,11,11,3.384184,12,0,1


New shape: (19950, 13)


In [9]:
# Check constant features
print("Constant columns:")

constant_cols = [
    col for col in processed_df.columns
    if processed_df[col].nunique() <= 1
]

print(constant_cols)

# Check suspicious word feature
print("\nSuspicious word distribution:")
print(processed_df["has_suspicious_words"].value_counts())

# Check label distribution
print("\nLabels:")
print(processed_df["label"].value_counts())

# Check missing values
print("\nMissing values:")
print(processed_df.isnull().sum())

# Check infinite values
print("\nInfinite values:")
print(
    processed_df.select_dtypes(include="number")
    .isin([float("inf"), float("-inf")])
    .sum()
)

Constant columns:
['special_chars']

Suspicious word distribution:
has_suspicious_words
0    19600
1      350
Name: count, dtype: int64

Labels:
label
1    10000
0     9950
Name: count, dtype: int64

Missing values:
length                  0
dots                    0
hyphens                 0
digits                  0
special_chars           0
subdomains              0
vowel_count             0
consonant_count         0
unique_chars            0
entropy                 0
longest_label           0
has_suspicious_words    0
label                   0
dtype: int64

Infinite values:
length                  0
dots                    0
hyphens                 0
digits                  0
special_chars           0
subdomains              0
vowel_count             0
consonant_count         0
unique_chars            0
entropy                 0
longest_label           0
has_suspicious_words    0
label                   0
dtype: int64


In [10]:
final_df = pd.concat([
    df[["domain"]].reset_index(drop=True),
    processed_df.reset_index(drop=True)
], axis=1)

final_df.to_csv("processed_domains.csv", index=False)

print("Saved successfully!")
print("Shape:", final_df.shape)

Saved successfully!
Shape: (19950, 14)


In [11]:
print("Final dataset:", final_df.shape)

print("Missing values:", final_df.isnull().sum().sum())

print("Duplicate domains:", final_df["domain"].duplicated().sum())

print("Label distribution:")
print(final_df["label"].value_counts())

print("Constant features:")
print([
    c for c in processed_df.columns
    if c != "label" and processed_df[c].nunique() <= 1
])

Final dataset: (19950, 14)
Missing values: 0
Duplicate domains: 0
Label distribution:
label
1    10000
0     9950
Name: count, dtype: int64
Constant features:
['special_chars']


In [12]:
X = final_df.drop(
    columns=["domain", "special_chars", "label"]
)

y = final_df["label"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (19950, 11)
y shape: (19950,)
